In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("./dfs/final_df.csv")
df.sample(3)

,question,evidence,ground_truth,url,source
14,During the constrained clustering procedure fo...,"[""Forming Clusters for Positive Sampling The g...",The system partitions the graph into disjoint ...,https://aclanthology.org/2022.naacl-main.343.pdf,1
12,How does the arborescence-based training model...,"[""Graph-based Dissimilarity Let Gbe a graph wi...",The model calculates dissimilarity using a min...,https://aclanthology.org/2022.naacl-main.343.pdf,1
17,What BLEU score did the baseline model develop...,"[""Table 2: The Transformer achieves better BLE...","The Transformer base model achieved 27.3 BLEU,...",attention is all you need & RoPE,2


## Querying

In [6]:
import importlib
import main
from datetime import datetime

importlib.reload(main)
run_pipeline = main.run_pipeline

db_dir = {
    "1": "./outputs/test1/",
    "2": "./outputs/test2/",
    "3": "./outputs/test3/"
}

In [9]:
start = datetime.now()
all_answers = []

for db_id, input_dir in db_dir.items():
    print(f"Running pipeline for DB {db_id}...")
    rows = df[df["source"] == int(db_id)]

    for _, row in rows.iterrows():
        question = row["question"]
        print(f"Question: {question}")
        answer, retrieved_content, decision = run_pipeline(query=question, vector_db_path=input_dir, top_k=3, is_parallel=True)
    
        all_answers.append({
                "question": question,
                "answer": answer,
                "context": retrieved_content,
                "ground_truth": row["ground_truth"],
                "evidence": row["evidence"],
                "source": db_id
            })
end = datetime.now()
ellapsed_time = end - start
print(f"Total time taken: {ellapsed_time}")

Running pipeline for DB 1...
Question: What is the target KB size used for MedMentions in the experiments?
Question: Do the training and inference batches contain mentions from the same document to leverage coreferent mentions as much as possible?


c:\Users\zakar\Desktop\RAG_nHop\venv\Lib\site-packages\matplotlib\transforms.py:195: RuntimeWarning: coroutine 'execute_rag_tree_parallel' was never awaited
  ref = weakref.ref(


Tree PNG saved -> decompositions\test2\decomposition_Do_the_training_and__aef8a91e-f29b-416e-98b6-d44f0f356dbb.png
Question: Does the model need to compute \psi(m_i, e) for all entities in the target knowledge base during inference?
Tree PNG saved -> decompositions\test2\decomposition_Does_the_model_need__5d2f53a6-25ab-41e2-924c-e573b2a2245a.png
Question: Why not use the complete graph for training instead of pruning it?
Tree PNG saved -> decompositions\test2\decomposition_Why_not_use_the_comp_e0663c5a-7eb6-4ab8-a9c1-ece3f1c8cb4b.png
Question: How does the arborescence-based training model calculate the dissimilarity between a mention and an entity, and what specific underlying models compute the edge weights for this calculation?
Tree PNG saved -> decompositions\test2\decomposition_How_does_the_arbores_5e88b513-1685-4ba2-bd35-787a8bce8332.png
Question: Why do the proposed arborescence-based cross-encoder models show consistent gains in linking accuracy on the MedMentions dataset but m

In [13]:
print(f"{(ellapsed_time/86).seconds} seconds per question on average")

11 seconds per question on average


## Evaluation

In [18]:
all_answers_df = pd.DataFrame(all_answers)
all_answers_df.to_csv("all_answers_2.csv", index=False)

In [19]:
all_answers_df.sample(2)

,question,answer,context,ground_truth,evidence,source
9,What BLEU score did the baseline model develop...,- Vaswani baseline BLEU on WMT-14 EN-DE: 27.3\...,[page_content='We choose the standard WMT 2014...,"The Transformer base model achieved 27.3 BLEU,...","[""Table 2: The Transformer achieves better BLE...",2
18,Is the dataset under a Creative Commons license?,I can’t tell from that list alone. License ter...,"[page_content='[1] Jia Deng, Wei Dong, Richard...",Yes.,"['Specifically , we will use Attribution 4.0 I...",3


In [20]:
def f1_token_overlap(pred, truth):
    pred_tokens = set(pred.lower().split())
    truth_tokens = set(truth.lower().split())
    if not pred_tokens or not truth_tokens:
        return 0
    return 2 * len(pred_tokens & truth_tokens) / (len(pred_tokens) + len(truth_tokens))

all_answers_df['f1'] = all_answers_df.apply(
    lambda row: f1_token_overlap(row['answer'], row['ground_truth']), axis=1
)

In [22]:
from rag_eval import evaluate

results = []
for row in all_answers_df.itertuples():
    print(f"Evaluating Question: {row.question}")
    result = evaluate(
        question=row.question,
        answer=row.answer,
        contexts=[doc.page_content for doc in row.context],
        ground_truth=row.ground_truth
    )
    results.append(result)

Evaluating Question: What is the target KB size used for MedMentions in the experiments?
Evaluating Question: Do the training and inference batches contain mentions from the same document to leverage coreferent mentions as much as possible?
Evaluating Question: Does the model need to compute \psi(m_i, e) for all entities in the target knowledge base during inference?
Evaluating Question: Why not use the complete graph for training instead of pruning it?
Evaluating Question: How does the arborescence-based training model calculate the dissimilarity between a mention and an entity, and what specific underlying models compute the edge weights for this calculation?
Evaluating Question: Why do the proposed arborescence-based cross-encoder models show consistent gains in linking accuracy on the MedMentions dataset but mixed results compared to mention-entity baselines on the ZeShEL dataset, and what specific statistical difference accounts for this?
Evaluating Question: During the constraine

In [23]:
df_with_eval = all_answers_df.copy()

In [24]:
for i, result in enumerate(results):
    for metric_name, metric_value in result.items():
        if isinstance(metric_value, dict):
            df_with_eval.at[i, metric_name] = metric_value.get("score")
        else:
            df_with_eval.at[i, metric_name] = metric_value

In [27]:
df_with_eval.to_csv("all_answers_with_eval_2.csv", index=False)

In [61]:
metrics = ["faithfulness", "context_relevance", "answer_relevance", "answer_correctness", "overall"]

for id in df_with_eval["source"].unique():
    curr = df_with_eval[df_with_eval["source"] == id]
    
    print(f"\n--- Mean Metrics for source {id} ---")
    print(curr[metrics].mean())


--- Mean Metrics for source 1 ---
faithfulness          1.000
context_relevance     0.850
answer_relevance      0.975
answer_correctness    0.875
overall               0.925
dtype: float64

--- Mean Metrics for source 2 ---
faithfulness          0.833333
context_relevance     0.600000
answer_relevance      0.900000
answer_correctness    0.833333
overall               0.791667
dtype: float64

--- Mean Metrics for source 3 ---
faithfulness          0.9500
context_relevance     0.6000
answer_relevance      0.9875
answer_correctness    0.5625
overall               0.7750
dtype: float64


### Parallel Computing

In [26]:
metrics = ["faithfulness", "context_relevance", "answer_relevance", "answer_correctness", "overall"]

for id in df_with_eval["source"].unique():
    curr = df_with_eval[df_with_eval["source"] == id]
    
    print(f"\n--- Mean Metrics for source {id} ---")
    print(curr[metrics].mean())


--- Mean Metrics for source 1 ---
faithfulness          1.00000
context_relevance     0.85000
answer_relevance      1.00000
answer_correctness    0.62500
overall               0.86875
dtype: float64

--- Mean Metrics for source 2 ---
faithfulness          1.000
context_relevance     0.900
answer_relevance      1.000
answer_correctness    1.000
overall               0.975
dtype: float64

--- Mean Metrics for source 3 ---
faithfulness          0.937500
context_relevance     0.625000
answer_relevance      0.950000
answer_correctness    0.375000
overall               0.721875
dtype: float64
